In [13]:
import torch
import itertools
from PIL import Image

# Your model loading function
from basemodel import load_dino_with_transforms, load_finetuned_timm_wrapper

# Zennit imports
from zennit.image import imgify
from zennit.composites import LayerMapComposite
import zennit.rules as z_rules

# LXT and Python operator imports
import operator
import torch.nn as nn
import torch.nn.functional as F
from lxt.explicit.core import Composite
import lxt.explicit.functional as lf
import lxt.explicit.modules as lm
import lxt.explicit.rules as rules

from lxt.explicit.modules import INIT_MODULE_MAPPING

In [14]:
class DinoAttentionLRPWrapper(lm.MultiheadAttention_CP):
    """
    A wrapper to solve the API mismatch between DINOv2's self-attention
    module and lxt's generic MultiheadAttention_CP.
    
    - DINOv2's attention is called with a single input: `attn(x)`
    - LXT's attention expects three inputs: `attn(query, key, value)`
    
    This wrapper provides a `forward(self, x)` method that calls the
    parent LXT module's forward pass with `query=x, key=x, value=x`.
    """
    def forward(self, x):
        # In self-attention, query, key, and value are all derived from the same input tensor `x`.
        # We call the parent class's (MultiheadAttention_CP) forward method with the correct signature.
        return super().forward(query=x, key=x, value=x)

In [18]:
def initialize_dino_attention(original, replacement_class):
    # This function is now PROVEN correct by the DINOv2 source code.
    print(f"Running CUSTOM initializer for {type(original).__name__}...")
    # ... (implementation is correct and remains unchanged) ...
    replacement_instance = replacement_class()
    embed_dim = original.qkv.in_features
    q_proj_weight, k_proj_weight, v_proj_weight = original.qkv.weight.chunk(3, dim=0)
    if original.qkv.bias is not None:
        q_proj_bias, k_proj_bias, v_proj_bias = original.qkv.bias.chunk(3, dim=0)
    else:
        q_proj_bias, k_proj_bias, v_proj_bias = None, None, None
    replacement_instance.q_proj_weight = q_proj_weight
    replacement_instance.k_proj_weight = k_proj_weight
    replacement_instance.v_proj.weight = v_proj_weight
    replacement_instance.bias_q = q_proj_bias
    replacement_instance.bias_k = k_proj_bias
    replacement_instance.v_proj.bias = v_proj_bias
    replacement_instance.out_proj.weight = original.proj.weight
    replacement_instance.out_proj.bias = original.proj.bias
    replacement_instance.embed_dim = embed_dim
    replacement_instance.num_heads = original.num_heads
    replacement_instance.head_dim = embed_dim // original.num_heads
    replacement_instance.batch_first = True
    return replacement_instance

In [20]:
# --- Setup ---
INIT_MODULE_MAPPING[DinoAttentionLRPWrapper] = initialize_dino_attention

CHECKPOINT_PATH = "/workspaces/vast-gorilla/gorillawatch/data/models/max_checkpoints/saved_checkpoints/giantbodybest74ens82.pth"
IMG_SIZE = 518
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BACKBONE = "vit_giant_patch14_dinov2.lvd142m"
EMBEDDING_DIM = 256 # The output size you trained for



model_wrapper, weights = load_finetuned_timm_wrapper(
    checkpoint_path=CHECKPOINT_PATH,
    backbone_name=BACKBONE,
    embedding_size=EMBEDDING_DIM,
    image_size=IMG_SIZE,
    device=DEVICE,
)



# 2. Programmatically get the specific class types from YOUR loaded model
DINO_ATTENTION_CLASS = type(model_wrapper.model.blocks[0].attn)
DINO_MLP_CLASS = type(model_wrapper.model.blocks[0].mlp)
DINO_LAYERSCALE_CLASS = type(model_wrapper.model.blocks[0].ls1)


# --- FIX: MONKEY-PATCH THE PROBLEMATIC FORWARD METHOD ---
# This bypasses the 'if not XFORMERS_AVAILABLE' and 'if self.training'
# control flow which is now confirmed to be the cause of the TracerError.
original_forward = DINO_ATTENTION_CLASS.forward
def tracer_friendly_forward(self, x):
    # This simplified forward pass is safe for the tracer.
    return self.proj(x)
DINO_ATTENTION_CLASS.forward = tracer_friendly_forward

# 3. Define the custom LRP composite tailored specifically for DINOv2
dinov2_attnlrp = Composite({
    # --- DINOv2 Specific Module Mappings ---
    # For DINOv2's custom `MemEffAttention` module. We replace it with the generic
    # LRP-aware ViT attention module provided by lxt.
    DINO_ATTENTION_CLASS: DinoAttentionLRPWrapper,
    
    # For DINOv2's `SwiGLUFFNFused` MLP. Since there's no custom LRP module for this,
    # we apply the general-purpose EpsilonRule, treating it as a single non-linear block.
    DINO_MLP_CLASS: rules.EpsilonRule,
    
    # For the standard `nn.LayerNorm` used by DINOv2. We replace it with the purpose-built
    # LRP-aware LayerNorm module from lxt.
    nn.LayerNorm: lm.LayerNormEpsilon,
    
    # For DINOv2's `LayerScale` module. It's a simple scaling, so the IdentityRule,
    # which passes relevance through unchanged, is the correct choice.
    DINO_LAYERSCALE_CLASS: rules.IdentityRule,
    
    # For DINOv2's `DropPath` (which is `nn.Identity` at inference).
    # The IdentityRule correctly passes relevance through.
    nn.Identity: rules.IdentityRule,

    # --- Rules for LRP-replacement modules themselves (from lm.MultiheadAttention_CP) ---
    # These are needed to configure the *internals* of the replacement attention module.
    lm.LinearInProjection: rules.EpsilonRule,
    lm.LinearOutProjection: rules.EpsilonRule,
    
    # --- Generic Functional Rules (to be replaced by the tracer) ---
    # For residual connections (hidden_states = residual + hidden_states)
    operator.add: lf.add2,
    # For any other matrix multiplications found by the tracer
    operator.matmul: lf.matmul,
    # For any other normalization functions found by the tracer
    F.normalize: lf.normalize,
})


--- Loading Finetuned TimmWrapper Model ---


INFO:basemodel:Setting img_size to 518


Building model architecture with backbone 'vit_giant_patch14_dinov2.lvd142m' and embedding size 256...


INFO:timm.models._builder:Loading pretrained weights from Hugging Face hub (timm/vit_giant_patch14_dinov2.lvd142m)
INFO:timm.models._hub:[timm/vit_giant_patch14_dinov2.lvd142m] Safe alternative available for 'pytorch_model.bin' (as 'model.safetensors'). Loading weights using safetensors.
INFO:basemodel:No pooling layer reset necessary.


State dict loading message: <All keys matched successfully>
Associated model transforms created successfully.
--- Model loading complete ---


In [23]:
# 4. Trace the model using the CORRECT dummy input format
dummy_tensor = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=DEVICE)

# The key 'x' corresponds to the name of the input argument in the
# DINOv2 model's forward method (def forward(self, x): ...)
dummy_inputs_dict = {'x': dummy_tensor}

print("\nTracing the DINOv2 model with our custom AttnLRP composite...")
# Pass the dictionary, not the raw tensor
try:
    traced_model = dinov2_attnlrp.register(model_wrapper.model, dummy_inputs=dummy_inputs_dict, verbose=True)
finally:
    # Restore the original method
    DINO_ATTENTION_CLASS.forward = original_forward
print("Tracing complete.")

# Load and preprocess the real input image
image = Image.open("/workspaces/bachelor_thesis_code/src/bachelor_thesis/image.png").convert("RGB")
input_tensor = weights(image).unsqueeze(0).to(DEVICE)

heatmaps = []

# 5. Run the LRP loop with the fine-tuning zennit composite
for conv_gamma, lin_gamma in itertools.product([0.1, 0.25, 100], [0, 0.01, 0.05, 0.1, 1]):
    input_tensor.grad = None
    print("\nGamma Conv2d:", conv_gamma, "Gamma Linear/MLP:", lin_gamma)

    # This zennit composite fine-tunes the rules set by the main dinov2_attnlrp tracer
    zennit_comp = LayerMapComposite([
        (nn.Conv2d, z_rules.Gamma(conv_gamma)),
        (nn.Linear, z_rules.Gamma(lin_gamma)),
        # Also apply the gamma rule to the entire MLP block
        (DINO_MLP_CLASS, z_rules.Gamma(lin_gamma)),
    ])
    zennit_comp.register(traced_model)

    y = traced_model(input_tensor.requires_grad_())
    target_scalar = y.sum()
    target_scalar.backward()
    zennit_comp.remove()

    heatmap = (input_tensor * input_tensor.grad).sum(1)
    heatmap = heatmap / abs(heatmap).max()
    heatmaps.append(heatmap[0].detach().cpu().numpy())

# Visualize the final heatmaps
imgify(heatmaps, vmin=-1, vmax=1, grid=(3, 5)).save('vit_heatmap_fast.png')


Tracing the DINOv2 model with our custom AttnLRP composite...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTOM initializer for Attention...
Running CUSTO

TraceError: symbolically traced variables cannot be used as inputs to control flow